# 08 · Capstone: ShipFast and the research agent in CrewAI

Notebooks 01 to 07 built the pieces. This notebook puts them together into the two agents every
framework in this project implements against the same contract (`contracts/`), and shows how they are
evaluated. The agents live in `crew_agents/`, and `run_eval.py` runs the full evaluation with the same code.

**You will see**

- the ShipFast agent's design in CrewAI, and why its Resolution is built in code
- two hard ShipFast scenarios run live through the **shared runner**: an approval, and a two-turn case
- one research question through the shared research runner
- this framework's full results, and all three frameworks side by side

In [1]:
# The agent module, its research workflow, and the adapter the shared runner drives.
import crew_agents.shipfast as sf  # import crew_agents first: telemetry and panels off
from crew_agents.research_flow import answer
from crew_agents.shipfast import CrewAIShipFast

# The shared harness from contracts/: the model check, the two runners and the test data.
from agent_contracts.harness import verify_model
from agent_contracts.harness.runners import run_question, run_scenario
from agent_contracts.research.scoring import load_questions
from agent_contracts.shipfast.scoring import load_scenarios

verify_model()  # stops with the fix if Ollama is not running or the shared model is missing
# Index the scenarios and questions by id, so a cell can pick one by name.
scenarios = {s["id"]: s for s in load_scenarios()}
questions = {q["id"]: q for q in load_questions()}

## The ShipFast agent

In CrewAI the agent is a **Flow**, one run per customer turn: triage, a router, a fixed decline or the specialist agent with
the gated refund, and a `@human_feedback` step that pauses with the state saved by `@persist`. The
conversation history travels in the flow's state from turn to turn.

Every build fills in the scored `Resolution` the same way, with the shared
`agent_contracts.shipfast.resolution.derive_resolution` (DECISIONS.md 16): `intent` and `order_id` from
triage; `action`, `refund_amount` and `ticket_id` from what actually happened in the ledger and the
tickets; and only when a turn left no record does the model choose between `answered`, `needs_info` and
`declined`. The component notebooks showed why: asked to fill in a Resolution itself, the 8B model
claimed refunds that never happened and invented ticket numbers.

In [2]:
# The module docstring in crew_agents/shipfast.py describes the agent's design. It is shown here because
# the file is the one place that states the whole flow in a few lines.
print(sf.__doc__)

The ShipFast support agent in CrewAI (ShipFast SPEC.md), the capstone of notebooks 01 to 07.

One Flow run per customer turn; the conversation history travels in the flow's state from turn to turn.

    triage (router) --"out_of_scope"--> send_decline
                    \--"in_scope"----> run_specialist (router) --"finished"--> done
                                                               \--"needs_approval"--> ask_supervisor
                                                                    (@human_feedback: pauses, state saved)
                                                                    --"approve"--> issue_approved_refund (code)
                                                                    --"reject"---> escalate_rejected_refund (agent)

The approval gate works as in notebook 06 (crew_agents/approval.py): the specialist's `issue_refund` is
gated above the shared $100 limit, the flow pauses between steps, and it resumes only from saved state.
The Resolution is b

The docstring draws the flow. In CrewAI each box is a method on the `ShipFastFlow` class, so you can check the diagram against the real thing by listing which of those steps the class has.

In [3]:
# Each flow step is a method on ShipFastFlow (in crew_agents/shipfast.py). List the ones that exist.
steps = ("triage", "route", "send_decline", "run_specialist", "check_refund", "ask_supervisor",
         "issue_approved_refund", "escalate_rejected_refund")
print("Steps found on ShipFastFlow:")
for step in steps:
    if hasattr(sf.ShipFastFlow, step):
        print(f"  {step}")

Steps found on ShipFastFlow:
  triage
  route
  send_decline
  run_specialist
  check_refund
  ask_supervisor
  issue_approved_refund
  escalate_rejected_refund


## Run through the shared runner

`run_scenario` is the same function every framework's `run_eval.py` uses. It resets the shared tools,
feeds each turn, resumes a paused approval with the scenario's decision, and checks the result against
`scenarios.json`. The framework only supplies the adapter: start a conversation, send a turn, resume.

`show()` turns a result into something you can read. It prints a summary block (verdict, cost and tools), a table with one row per customer turn (the `Resolution` the agent produced for that turn: intent, order, action and refund amount) with the start of the reply under it, and then any **problems**, which are the checks from `scenarios.json` that failed.

In [4]:
# show() prints one scenario result: a summary block, a table with one row per turn, then any problems.
def show(item):
    print(f"Scenario:    {item['id']}")
    print(f"Result:      {'PASS' if item['passed'] else 'FAIL'}")  # PASS means every check in scenarios.json held
    print(f"Cost:        {item['llm_calls']} model calls in {item['latency_s']:.0f}s")
    print(f"Tools used:  {', '.join(item['tool_calls']) or 'none'}")  # in the order the agent called them
    print()
    print(f"{'Turn':4}  {'Intent':15} {'Order':6} {'Action':11} {'Refund':>7}")
    for i, t in enumerate(item["turn_resolutions"], 1):
        refund = t["refund_amount"] or "-"  # "-" when no money moved
        print(f"{i:<4}  {t['intent']:15} {t['order_id'] or '-':6} {t['action']:11} {refund:>7}")
        print(f"      reply: {t['reply'].strip()[:160]}")  # the first 160 characters of what the customer saw
    print()
    if not item["problems"]:
        print("Problems:    none")
    for p in item["problems"]:
        print("Problem:    ", p)  # one check from scenarios.json that failed

First scenario: S3a, a late order that needs a refund above the approval limit.

In [5]:
# Scenario S3a: the late espresso machine, a $250 refund that needs approval.
# run_scenario plays the scenario's turns against a fresh agent and returns the scored result as a dict.
result = await run_scenario(CrewAIShipFast(), scenarios["S3a_late_refund_approved"])
show(result)

╭──────────────────────────────────────────────── ⏳ Flow Paused ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Paused - Waiting for Feedback                                                                             │
│  Name: ShipFastFlow                                                                                             │
│  ID: d71e6e49-a89d-4484-9754-3886b61318e2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Scenario:    S3a_late_refund_approved
Result:      PASS
Cost:        7 model calls in 48s
Tools used:  lookup_order, track_shipment, search_policy, issue_refund

Turn  Intent          Order  Action       Refund
1     refund_request  A1003  refunded      250.0
      reply: Good news: your refund of $250.00 for order A1003 has been approved and issued.

Problems:    none


S3a is the late espresso machine, a $250 refund the supervisor approves. A passing run pauses at the
approval gate, is resumed with `approve`, and issues exactly $250.

In [6]:
# Scenario S4: the broken vase, a two-turn conversation.
# run_scenario plays the scenario's turns against a fresh agent and returns the scored result as a dict.
result = await run_scenario(CrewAIShipFast(), scenarios["S4_damaged_needs_photo_then_refund"])
show(result)

Scenario:    S4_damaged_needs_photo_then_refund
Result:      FAIL
Cost:        12 model calls in 79s
Tools used:  lookup_order, track_shipment, search_policy, issue_refund, lookup_order, search_policy, issue_refund

Turn  Intent          Order  Action       Refund
1     complaint       A1004  refunded       60.0
      reply: The refund for your order A1004 has been issued. Please provide a photo of the damaged vase set so we can process the refund. Thank you for your understanding.
2     complaint       A1004  answered          -
      reply: The refund for your order A1004 has already been processed in full. If you need further assistance, please let me know. Thank you for your understanding.

Problem:     action: expected 'refunded', got 'answered'
Problem:     refund_amount: expected 60.0, got None
Problem:     turn 1: action: expected 'needs_info', got 'refunded'
Problem:     turn 1: refund_amount: expected None, got 60.0
Problem:     turn 2: action: expected 'refunded', got 'answe

S4 is the broken vase: the first turn must ask for a photo and refund nothing (policy P3); the second,
which never names the order, must refund the $60 total. It needs memory and policy in the right order.

When we ran them, S3a passed: the gated refund recorded $250 as pending, the flow paused at
`@human_feedback`, was restored from saved state and resumed with `approve`, and code issued exactly $250.
S4 failed the way it did in notebooks 03 and 04: the specialist refunded $60 on the first turn, before
any photo, while telling the customer to send one. On the second turn the shared `issue_refund` refused a
second refund (DECISIONS.md 15), so the ledger holds one $60, but the money moved before policy P3 allowed
it, and the scenario fails.

## One research question

The research agent is the workflow from notebook 05, answered through the shared `run_question`. Q07 has
a stale article (9 provinces) beside the current one (14 provinces); mentioning the stale figure at all
fails the answer.

In [7]:
# Q07 has a stale article beside the current one. `answer` is this tutorial's research workflow, and
# run_question runs it on one question and returns the scored result as a dict.
item = await run_question(answer, questions["Q07_deployment"])

The result is a dict. Print its fields with a label each.

In [8]:
# Print the parts of the result worth reading, one labelled line each.
print("Passed:      ", item["passed"])  # True when the answer holds every expected fact and no stale one
print("Tools used:  ", ", ".join(item["tool_calls"]) or "none")
print("Model calls: ", item["llm_calls"])
print("Time:        ", f"{item['latency_s']:.0f}s")
print("Answer:      ", item["report"]["answer"])
print("Problems:    ", "; ".join(item["problems"]) or "none")  # what the scorer objected to, if anything

Passed:       True
Tools used:   web_search, web_search, web_search, web_search, web_search, web_search
Model calls:  10
Time:         141s
Answer:       Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces, according to the 2025 annual report [F04].
Problems:     none


## The full evaluation

`run_eval.py` runs every scenario and every question once and writes `results/shipfast/crewai.json` and
`results/research/crewai.json` (from the tutorial folder: `uv run python run_eval.py`). It takes about half an
hour on a 16 GB M4 Mac, so here we read the saved results instead of re-running them.

The table below has one row per scenario or question: its id, whether it passed, and how many seconds it took. Under a failing row, each `problem` is one check the scorer reported.

In [9]:
import json
from pathlib import Path

# The saved results live in results/<agent>/<framework>.json at the repo root, one level above the tutorial folder.
RESULTS = Path("..").resolve().parent / "results"


# Read one results file; returns None if that framework has not been run yet.
def load(agent, framework):
    path = RESULTS / agent / f"{framework}.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None

In [10]:
# One block per agent: a summary line, then a table with one row per scenario or question.
for agent, key in (("shipfast", "scenarios"), ("research", "questions")):
    data = load(agent, "crewai")
    summary = data["summary"]
    print(f"{agent}: {summary['passed']} of {summary['total']} passed (run at {data['run_at']})")
    print(f"  {'id':40} {'result':6} {'seconds':>7}")
    for it in data[key]:
        print(f"  {it['id']:40} {'PASS' if it['passed'] else 'FAIL':6} {it['latency_s']:7.0f}")
        for problem in it["problems"]:  # only failing items have problems
            print(f"      problem: {problem}")
    print()

shipfast: 7 of 8 passed (run at 2026-11-03T08:34:31+00:00)
  id                                       result seconds
  S1_status_with_followup                  PASS        70
  S2_lost_refund_auto                      PASS        51
  S3a_late_refund_approved                 PASS        59
  S3b_late_refund_rejected                 PASS        79
  S4_damaged_needs_photo_then_refund       FAIL        82
      problem: action: expected 'refunded', got 'answered'
      problem: refund_amount: expected 60.0, got None
      problem: turn 1: action: expected 'needs_info', got 'refunded'
      problem: turn 1: refund_amount: expected None, got 60.0
      problem: turn 2: action: expected 'refunded', got 'answered'
      problem: turn 2: refund_amount: expected 60.0, got None
  S5_outside_refund_window                 PASS        38
  S6_unknown_order                         PASS        19
  S7_out_of_scope                          PASS         3

research: 9 of 10 passed (run at 2026-11-03T0

## All three frameworks

The same contract, data, model, settings, tools, instructions, scoring and runner, so the differences
below come from the frameworks (and from how a small model reacts to each one's way of packaging the
same request).

In the table, `passed` is passed over total, `tokens` is the total tokens the model read and wrote, and `seconds` is the total run time, for that agent in that framework.

In [11]:
# Collect one row of totals per agent and framework, then print them as a table.
rows = []
for agent in ("shipfast", "research"):
    for framework in ("langgraph", "crewai", "strands"):
        data = load(agent, framework)
        if data:  # skip a framework whose results file does not exist yet
            s = data["summary"]
            rows.append((agent, framework, f"{s['passed']}/{s['total']}", s["total_tokens"], round(s["total_latency_s"])))

print(f"{'agent':9} {'framework':10} {'passed':7} {'tokens':>7} {'seconds':>8}")
for agent, framework, passed, tokens, seconds in rows:
    print(f"{agent:9} {framework:10} {passed:7} {tokens:7} {seconds:8}")

agent     framework  passed   tokens  seconds
shipfast  langgraph  7/8       29962      387
shipfast  crewai     7/8       34867      401
shipfast  strands    4/8       26981      331
research  langgraph  10/10     44361     1464
research  crewai     9/10      60243     1194
research  strands    9/10      48803     1147


CrewAI passed 7 of 8 ShipFast scenarios and 9 of 10 research questions. The ShipFast failure is S4 above.
The research failure, Q10, gave the correct total (PHP 1,635,000) and both subtotals but not the two unit
prices, which the question's expected facts include; whether the check should require them is an open
question for the contract.

Across the frameworks, CrewAI ties LangGraph on ShipFast with a different single failure, and used the
most tokens on both agents: each CrewAI agent call carries a longer prompt (role, goal, backstory), and its
research agent searched more often. These are single runs on one laptop: a first measurement, not a
ranking.

## How the others do this

- **LangGraph** builds the same agent as a `StateGraph` with the specialist as a subgraph; the approval
  pause happens inside the tool call, and a checkpointer holds the conversation and the pause.
- **Strands** uses **one agent per conversation**, with triage as a separate structured call and the
  approval as a hook that interrupts before the refund.

## Takeaways

- The contract keeps the comparison fair: one copy of the data, tools, instructions, scoring and runner.
- Code records the facts (refunds, tickets, pauses); the model writes the reply and makes one small call.
- Every framework could build both agents, but each needed its own shape for the approval pause and for
  memory, and each made different mistakes with the same model. Those differences are the comparison.